# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and analyze the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source (Croissant schema):
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")

## 2. Data Overview
Review available record sets, their fields, and column names. All entities are referenced using their `@id` values, per Croissant best practices.

In [ ]:
# List all record sets defined by @id

from pprint import pprint

def get_record_sets(ds):
    """Return a list of record set objects with their @id."""
    if hasattr(ds.metadata, 'record_sets'):
        # Newer mlcroissant API 
        record_sets = ds.metadata.record_sets
    elif hasattr(ds.metadata, 'recordSet'):
        # fallback for potential legacy
        record_sets = ds.metadata.recordSet
    else:
        # fallback for field name difference
        record_sets = getattr(ds.metadata, 'recordset', [])
    return [rs for rs in record_sets]

# Try to enumerate record sets using the direct attribute
record_sets_objs = get_record_sets(dataset)

# There may be no explicit record sets in metadata (as per your metadata snippet),
# so let's try to discover them programmatically
if not record_sets_objs:
    record_sets_ids = []
    print("No record sets found directly in metadata. Attempting to discover data sources...")
    # Try iterating with dataset.record_set_ids()
    if hasattr(dataset, 'record_set_ids'):
        record_sets_ids = list(dataset.record_set_ids())
else:
    record_sets_ids = [getattr(rs, '@id', getattr(rs, 'id', str(idx))) for idx, rs in enumerate(record_sets_objs)]

# If none found yet, try dataset.records() for autodiscovery (most croissant datasets have at least one main set)
if not record_sets_ids:
    # Try .records() with no arguments to see what record_sets are available
    test_iter = dataset.records()
    try:
        peek = next(test_iter)
        print('Example record found:', peek)
        record_sets_ids = []  # Still unknown, but user can prompt for more info
    except Exception as e:
        print('No records autodiscovered:', e)

# If record_sets_ids still empty, fallback to documented ID (sometimes just one table with a known id)
# From live API testing, common pattern is main data in '@id': 'Patient', 'Table', etc.
if not record_sets_ids:
    # Try using all ids found in distribution or entries in the dataset schema
    print("Could not detect record set IDs automatically.")
    print("You may need to check the source schema for the main table '@id'.")
    record_sets_ids = []

if record_sets_ids:
    print("Record set @ids discovered:")
    for rec_id in record_sets_ids:
        print(f" - {rec_id}")
    
    # Try to print fields/columns for each record set
    for record_set_id in record_sets_ids:
        print(f"\nFields for record set {record_set_id}:")
        example_records = list(dataset.records(record_set=record_set_id))
        if example_records:
            first_rec = example_records[0]
            print(f"Column names (@ids):\n{list(first_rec.keys())}")
        else:
            print("No records found for this set.")
else:
    print("No record set IDs detected.")

## 3. Data Extraction
Load data from each available record set into Pandas DataFrames using their `@id`.

In [ ]:
# For this dataset, let's try loading the first found record set

if not record_sets_ids:
    print("No record set IDs available to extract data.")
    dataframes = {}  # Fallback empty
else:
    dataframes = {}
    for record_set_id in record_sets_ids:
        print(f"\nLoading data for record set: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns in {record_set_id}: {df.columns.tolist()}")
        display(df.head())  # Show top rows

    # Choose the first record set for analysis below
    selected_record_set = record_sets_ids[0]
    df_main = dataframes[selected_record_set]

## 4. Exploratory Data Analysis (EDA)
Demonstration of filtering, normalization, and grouping with the dataset fields, referencing all fields/columns by their `@id`.

In [ ]:
# ------- Setup for EDA -------

# For demonstration, try to find a numeric field in the selected DataFrame
numeric_field = None
group_field = None

if not dataframes:
    print("No dataframes loaded -- cannot perform EDA.")
else:
    sample_df = df_main
    # Try infer numeric columns (float or int)
    num_fields = [
        f for f in sample_df.columns
        if pd.api.types.is_numeric_dtype(sample_df[f]) and not sample_df[f].isnull().all()
    ]
    if not num_fields:
        # Fallback: try to convert any columns named 'age', 'interval', or containing numbers
        for f in sample_df.columns:
            if any(x in f.lower() for x in ['age', 'interval', 'score', 'number', 'count']):
                try:
                    sample_df[f] = pd.to_numeric(sample_df[f], errors='coerce')
                    if sample_df[f].notnull().any():
                        num_fields.append(f)
                except Exception:
                    pass
    
    if num_fields:
        numeric_field = num_fields[0]
        print(f"Selected numeric field (@id): {numeric_field}")
    else:
        print("No numeric field found.")

    # Pick a grouping field: try the first object/string column, not same as numeric
    obj_fields = [col for col in sample_df.columns if col != numeric_field and sample_df[col].dtype == 'object']
    if obj_fields:
        group_field = obj_fields[0]
        print(f"Selected group field (@id): {group_field}")

    # Example threshold filtering
    if numeric_field:
        threshold = sample_df[numeric_field].quantile(0.75)  # Use 75th percentile as threshold for demonstration
        filtered_df = sample_df[sample_df[numeric_field] > threshold].copy()
        print(f"\nFiltered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nFirst few normalized records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouping
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped by {group_field} (mean of {numeric_field}):")
            display(grouped_df.head())
    else:
        print("Numeric analysis not available (no numeric fields found).")

## 5. Visualization
Visualize the distribution and relationships between fields. Example: histogram of the numeric field, boxplots grouped by category, or scatter plots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes:
    print("No dataframes loaded for visualization.")
elif numeric_field is None:
    print("No numeric field selected for plots.")
else:
    plt.figure(figsize=(8,4))
    sns.histplot(df_main[numeric_field].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(y=df_main[numeric_field], x=df_main[group_field])
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, inspect, and conduct basic analysis of the FAIR^2 dataset using `mlcroissant`. All data access was performed via Croissant `@id` references to ensure schema alignment and reproducibility.

Further steps could include advanced statistical analysis, machine learning modeling, and integration with domain-specific biomedical workflows.